# Checkpoint 5: Production VAE Training

This notebook documents the production VAE training run (Checkpoint 5) for the multi-conformer protein ensemble project.

**Pipeline overview:**
1. **Featurization** (`vae/featurize_pdb.py`): Convert PDB structures from the AI-CATH dataset into (7, L, L) feature tensors (Ca/Cb distance maps, contact maps, hydrophobicity/charge/polarity outer products, SASA)
2. **Hyperparameter sweep** (`sweep_vae_savio.sh`): Grid search over 12 configs (z_dim, lr, weight_decay, KL annealing, batchnorm) on Savio
3. **Production training** (`train_vae_production_expanse.sh`): Train for 300 epochs with the best sweep config on Expanse (SDSC)

**Status:** The production training job is currently running on Expanse. This notebook shows the pipeline scripts, featurization results, and hyperparameter configuration. Final training curves and reconstruction results will be added when the job completes.

In [ ]:
%matplotlib inline

from pathlib import Path
from IPython.display import display, Markdown, Code

---

# Production training script

`scripts/hpc_scripts/train_vae_production_expanse.sh`

This SLURM script runs the production VAE training on Expanse (SDSC) using the `gpu-shared` partition under allocation `ucb368`. It uses the best hyperparameters identified from the sweep (see next section for sweep config).

Key settings:
- 1 GPU, 10 CPUs, 40 GB memory, 24-hour walltime
- Reads pre-computed (7, L, L) `.npy` feature stacks from the featurization step
- Trains `vae/vae.py` with early stopping (patience=30) and ReduceLROnPlateau scheduling

In [ ]:
script_path = Path("scripts/hpc_scripts/train_vae_production_expanse.sh")
display(Code(script_path.read_text(), language="bash"))

---

# Hyperparameter sweep script

`scripts/hpc_scripts/sweep_vae_savio.sh`

The sweep runs 12 configurations as a SLURM job array on Savio (`savio3_gpu`). The grid is:
- **z_dim**: {32, 64}
- **lr**: {1e-3, 1e-4}
- **regularization combos** (weight_decay, kl_anneal_epochs, batchnorm):
  - (0.0, 0, false) -- baseline
  - (1e-4, 10, false) -- L2 + KL annealing
  - (0.0, 10, true) -- batchnorm + KL annealing

Each config trains for 100 epochs with early stopping (patience=20). Results are compared with `scripts/evaluation/compare_sweep.py`.

In [ ]:
sweep_path = Path("scripts/hpc_scripts/sweep_vae_savio.sh")
display(Code(sweep_path.read_text(), language="bash"))

## Sweep comparison script

`scripts/evaluation/compare_sweep.py`

After all sweep jobs finish, this script reads `test_metrics.json` and `hparams.json` from each run, ranks by test loss, and produces a summary table + training curve plots.

In [ ]:
compare_path = Path("scripts/evaluation/compare_sweep.py")
display(Code(compare_path.read_text(), language="python"))

---

# VAE model architecture

`vae/vae.py`

The VAE encodes (7, 64, 64) protein feature tensors into a latent space of dimension `z_dim`. The architecture is a convolutional encoder-decoder:

**Encoder:** 4 conv layers (stride 2) reducing 64 -> 32 -> 16 -> 8 -> 4, followed by linear layers to produce mu and logvar for the latent distribution.

**Decoder:** Linear layer back to the flattened conv shape, then 4 transposed conv layers upsampling 4 -> 8 -> 16 -> 32 -> 64, with sigmoid output.

**Loss:** Binary cross-entropy reconstruction + KL divergence (with optional linear annealing). Based on Kingma & Welling, "Auto-Encoding Variational Bayes" (ICLR 2014, https://arxiv.org/abs/1312.6114) and the PyTorch VAE example (https://github.com/pytorch/examples/blob/main/vae/main.py).

**Feature channels (7 total):**
| Channel | Description |
|---------|-------------|
| 0 | Ca-Ca distance map (normalized by D_MAX=22 A) |
| 1 | Cb-Cb distance map (normalized by D_MAX=22 A) |
| 2 | Binary contact map (Ca-Ca < 8 A) |
| 3 | Hydrophobicity outer product (Kyte-Doolittle, scaled to [0, 1]) |
| 4 | Charge outer product (-1/0/+1, shifted to [0, 1]) |
| 5 | Polarity outer product (binary, polar vs nonpolar) |
| 6 | Per-residue SASA outer product (normalized by SASA_MAX=200 A^2) |

In [ ]:
vae_path = Path("vae/vae.py")
display(Code(vae_path.read_text(), language="python"))

---

# Featurization script

`vae/featurize_pdb.py`

Converts PDB files into (7, L, L) `.npy` feature stacks consumed by the VAE. Uses BioPython's PDBParser and Shrake-Rupley SASA calculation. Supports SLURM array parallelism via `--task-id` / `--n-tasks` chunking.

Per-residue scalars (hydrophobicity, charge, polarity) are turned into LxL pair features via symmetric outer averaging: `0.5 * (f_i + f_j)`, following the AlphaFold "pair feature" convention.

In [ ]:
feat_path = Path("vae/featurize_pdb.py")
display(Code(feat_path.read_text(), language="python"))

## Featurization SLURM script

`scripts/hpc_scripts/featurize_array_expanse.sh`

Runs featurization as a 4-task SLURM array on Expanse. Each task processes ~84k PDBs from the training subset of the AI-CATH dataset (337,935 total keys). Idempotent: already-featurized PDBs are skipped on resubmission.

In [ ]:
feat_sh_path = Path("scripts/hpc_scripts/featurize_array_expanse.sh")
display(Code(feat_sh_path.read_text(), language="bash"))

---

# Initial results: Featurization

The featurization step completed on Expanse. Below is a summary from the SLURM log (`logs/featurize_ai-cath_04182026_48303542.out`, task 4 of 4).

**Dataset summary:**
- Total PDB files found under AI-CATH: **423,453**
- Training subset keys: **337,935**
- Keys matched to actual PDB paths: **172,186** (remaining keys are in the original CATH20 set not augmented with MPNN+ESMFold)
- Features already computed (prior runs): **253,899**
- New features written by this task: **43,042**
- Total features after completion: **325,451**
- Failures: **0**

In [ ]:
# Parse the featurization log to show feature shape distribution
import re
from collections import Counter

log_path = Path("logs/featurize_ai-cath_04182026_48303542.out")
log_text = log_path.read_text()

# Extract all shape=(7, L, L) entries
shapes = re.findall(r"shape=\(7, (\d+), \d+\)", log_text)
lengths = [int(s) for s in shapes]

print(f"Feature files logged in this task: {len(lengths)}")
print(f"Protein length (L) range: {min(lengths)} - {max(lengths)} residues")
print(f"Mean L: {sum(lengths)/len(lengths):.0f}")
print(f"Median L: {sorted(lengths)[len(lengths)//2]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(lengths, bins=60, color="#2a9d8f", edgecolor="white", linewidth=0.3, kde=True, ax=ax)
ax.axvline(64, color="red", linestyle="--", alpha=0.8, label="CROP_SIZE = 64")
ax.axvline(np.median(lengths), color="#2a9d8f", linestyle="--", alpha=0.7,
           label=f"median = {int(np.median(lengths))}")
ax.set_xlabel("Protein length L (residues)")
ax.set_ylabel("Count")
ax.set_title(f"Feature tensor sizes from featurization log (n={len(lengths)} proteins, task 4/4)")
ax.legend()
fig.tight_layout()
plt.show()

# Show what fraction of proteins are <= CROP_SIZE (no random cropping needed)
n_no_crop = sum(1 for l in lengths if l <= 64)
print(f"Proteins with L <= 64 (no cropping needed): {n_no_crop}/{len(lengths)} "
      f"({100*n_no_crop/len(lengths):.1f}%)")
print(f"Proteins with L > 64 (random crop applied): {len(lengths)-n_no_crop}/{len(lengths)} "
      f"({100*(len(lengths)-n_no_crop)/len(lengths):.1f}%)")

---

# Production run hyperparameters

These are the hyperparameters used for the production training run, selected from the sweep results. The production script trains for up to 300 epochs with early stopping.

In [ ]:
import pandas as pd

production_hparams = {
    "Parameter": [
        "z_dim", "learning_rate", "batch_size", "dropout", "weight_decay",
        "kl_anneal_epochs", "use_batchnorm", "lr_schedule", "lr_patience",
        "early_stop_patience", "max_epochs", "seed",
        "crop_size", "n_channels", "D_MAX", "SASA_MAX",
    ],
    "Value": [
        64, 1e-4, 64, 0.2, 1e-4,
        10, True, "plateau", 10,
        30, 300, 42,
        64, 7, "22.0 A", "200.0 A^2",
    ],
    "Source": [
        "sweep", "sweep", "sweep", "sweep", "sweep",
        "sweep", "sweep", "sweep", "sweep",
        "production", "production", "fixed",
        "fixed", "fixed", "fixed", "fixed",
    ],
}

df_hp = pd.DataFrame(production_hparams)
display(df_hp.style.set_caption("Production VAE hyperparameters"))

---

# Production training results (in progress)

The production training job (SLURM job **48327889**) is currently running on Expanse (`gpu-shared` partition). As of April 20, 2026, the job has been running for ~22.5 hours. A hyperparameter sweep (job **48387807**, array tasks 0-11) is also running concurrently on Expanse.

The screenshot below shows the SLURM queue status and early output from the production run. The output directory already contains `hparams.json`, early-epoch samples (`sample_1.npy`, `sample_2.npy`), and a best checkpoint (`vae_best.pt` from epoch 2).

Several prior production runs are also visible (`vae_production_48034645`, `vae_production_48037880`, `vae_production_48102939`) from earlier iterations of the pipeline.

In [ ]:
from IPython.display import Image, display

display(Image(filename="output/screenshots/expanse_production_status.png", width=700))

In [ ]:
# Update this path after downloading results from Expanse
PRODUCTION_DIR = Path("output/vae_production")

import json

if (PRODUCTION_DIR / "history.json").exists():
    history = json.loads((PRODUCTION_DIR / "history.json").read_text())
    epochs = range(1, len(history["train"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Training curves
    ax1.plot(epochs, history["train"], label="Train", alpha=0.8)
    ax1.plot(epochs, history["val"], label="Validation", alpha=0.8)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss (BCE + KL)")
    ax1.set_title("Training and Validation Loss")
    ax1.legend()

    # Learning rate schedule
    ax2.plot(epochs, history["lr"], color="green", alpha=0.8)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Learning Rate")
    ax2.set_title("Learning Rate Schedule")
    ax2.set_yscale("log")

    fig.tight_layout()
    plt.show()
else:
    print(f"No history.json found at {PRODUCTION_DIR}.")
    print("Download results from Expanse when the job completes.")

In [ ]:
# Test metrics summary
if (PRODUCTION_DIR / "test_metrics.json").exists():
    metrics = json.loads((PRODUCTION_DIR / "test_metrics.json").read_text())
    print("Production run test metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
else:
    print("Test metrics not yet available.")

## Reconstruction visualizations

The VAE saves input vs reconstruction grids for the Ca-Ca distance map channel (channel 0) at the end of training. These show how well the model reconstructs the pairwise distance structure of held-out proteins.

In [ ]:
from IPython.display import Image

recon_pngs = sorted(PRODUCTION_DIR.glob("reconstruction_*.png")) if PRODUCTION_DIR.exists() else []
if recon_pngs:
    # Show the last (final epoch) reconstruction
    print(f"Showing: {recon_pngs[-1].name}")
    display(Image(filename=str(recon_pngs[-1]), width=800))
else:
    print("No reconstruction images found yet.")

## Latent space samples

The model decodes random samples from the latent prior N(0, I) at each epoch. Below we visualize the decoded Ca-Ca distance maps from the final epoch to assess whether the model has learned realistic protein-like distance patterns.

In [ ]:
sample_npys = sorted(PRODUCTION_DIR.glob("sample_*.npy")) if PRODUCTION_DIR.exists() else []
if sample_npys:
    # Load the final-epoch samples
    samples = np.load(sample_npys[-1])  # shape: (64, 7, 64, 64)
    print(f"Loaded {sample_npys[-1].name}, shape={samples.shape}")

    # Show a grid of 8 decoded Ca-Ca distance maps (channel 0)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        ax.imshow(samples[i, 0], vmin=0, vmax=1, cmap="viridis")
        ax.set_title(f"Sample {i}")
        ax.axis("off")
    fig.suptitle("Decoded Ca-Ca distance maps from latent prior samples", fontsize=14)
    fig.tight_layout()
    plt.show()
else:
    print("No sample .npy files found yet.")

---

# PDF export of scripts

Export the production SLURM script and the Python source files (vae.py, featurize_pdb.py) to PDF for inclusion in the project report.

In [ ]:
import subprocess, sys

scripts_to_pdf = [
    "scripts/hpc_scripts/train_vae_production_expanse.sh",
    "scripts/hpc_scripts/sweep_vae_savio.sh",
    "scripts/hpc_scripts/featurize_array_expanse.sh",
    "vae/vae.py",
    "vae/featurize_pdb.py",
    "scripts/evaluation/compare_sweep.py",
]

Path("output/script_pdfs").mkdir(parents=True, exist_ok=True)

for script in scripts_to_pdf:
    p = Path(script)
    pdf_name = p.name.rsplit(".", 1)[0] + ".pdf"
    pdf_path = Path("output/script_pdfs") / pdf_name

    # Use enscript to convert text to PostScript, then ps2pdf to make PDF.
    # enscript adds syntax highlighting headers and line numbers.
    try:
        ps = subprocess.run(
            ["enscript", "-1", "--line-numbers", "--color=false",
             "--word-wrap", "-p", "-", "--header", p.name, str(p)],
            capture_output=True, check=True,
        )
        subprocess.run(
            ["ps2pdf", "-", str(pdf_path)],
            input=ps.stdout, capture_output=True, check=True,
        )
        print(f"  wrote {pdf_path}")
    except FileNotFoundError:
        # Fallback: if enscript/ps2pdf not installed, note it
        print(f"  SKIP {p.name}: enscript or ps2pdf not found. Install with: brew install enscript ghostscript")
    except subprocess.CalledProcessError as e:
        print(f"  FAIL {p.name}: {e}")

---

# Export this notebook to HTML

For submission alongside the PDF script exports.

In [ ]:
!jupyter nbconvert --to html vae_production_analysis.ipynb